# Example 02 Lecture Note: Point-Cloud Bifiltration (`:rips_density`)

This notebook is a full dataset workflow: multiple samples, one unified feature table.

## Learning goals
- Build a deterministic two-class point-cloud dataset.
- Control ingestion growth with sparse construction and explicit budgets.
- Encode each sample and featurize with a fixed specification.
- Export outputs in a form suitable for downstream statistics.
        


## Modeling decisions and alternatives

Chosen here:
- `kind=:rips_density`, `max_dim=1`
- `sparsify=:knn`, `knn=10`, `density_k=10`
- feature specs: sliced barcode summaries + landscapes

Alternatives:
- `kind=:rips` for single-parameter geometric behavior
- `kind=:function_rips` when a domain-specific vertex signal is available
- different direction/offset grids for task-specific directional sensitivity
        


In [ ]:
# Load shared example helpers.
common_candidates = (
    joinpath(pwd(), "examples", "_common.jl"),
    joinpath(pwd(), "_common.jl"),
)

common_path = nothing
for p in common_candidates
    if isfile(p)
        common_path = p
        break
    end
end
common_path === nothing && error("Could not find examples/_common.jl. Start from repo root or examples/.")
include(common_path)

println("Loaded _common.jl from: ", common_path)

Random.seed!(20260217)
outdir = example_outdir("02_point_cloud_notebook")
println("outdir = ", outdir)


## Step 1: Build a deterministic two-class dataset


In [ ]:
# Use shared deterministic dataset helper from _common.jl.
clouds, labels = make_pointcloud_dataset(n_per_class=10, n_points=64, seed=2002)

println("dataset size = ", length(clouds))
println("circle count = ", count(==("circle"), labels))
println("fig8 count   = ", count(==("figure8"), labels))


## Step 2: Declare ingestion contracts and encode all samples


In [ ]:
construction = TO.ConstructionOptions(
    ;
    sparsify=:knn,
    collapse=:none,
    budget=(max_simplices=140_000, max_edges=90_000, memory_budget_bytes=1_000_000_000),
)

filtration = TO.RipsDensityFiltration(
    ;
    max_dim=1,
    knn=10,
    density_k=10,
    nn_backend=:auto,
    construction=construction,
)

field = TO.CoreModules.F2()
encodings = Vector{TO.EncodingResult}(undef, length(clouds))
for i in eachindex(clouds)
    encodings[i] = TO.encode(clouds[i], filtration; degree=0, field=field, cache=:auto)
end

println("encoded samples = ", length(encodings))
println("one enc type    = ", typeof(encodings[1]))


## Step 3: Inspect one `H^1` dimension plane before featurizing

Before flattening the whole dataset into features, it helps to inspect one dims-only bifiltration view on a representative sample. `stage=:cohomology_dims` is the cheap workflow path when only cohomology dimensions are needed, and `kind=:cohomology_support_plane` shows the nonzero `H^1` support directly on the parameter plane.


In [ ]:
demo_idx = findfirst(==("figure8"), labels)
demo_h1 = TO.encode(clouds[demo_idx], filtration; degree=1, stage=:cohomology_dims, field=field, cache=:auto)
demo_h1_spec = TO.visual_spec(demo_h1; kind=:cohomology_support_plane)

(; demo_result=TO.describe(demo_h1),
   demo_support_summary=TO.visual_summary(demo_h1_spec))


In [ ]:
TO.visualize(demo_h1; kind=:cohomology_support_plane, backend=:cairomakie)


## Step 4: Feature specification and invariant policy


In [ ]:
dirs = [[1.0, 0.0], [0.0, 1.0], [1.0, 1.0], [1.0, -1.0]]
offs = [
    collect(range(-0.6, stop=0.6, length=5)),
    collect(range(-0.6, stop=0.6, length=5)),
    collect(range(-0.8, stop=0.8, length=5)),
    collect(range(-0.8, stop=0.8, length=5)),
]

sbar_spec = TO.SlicedBarcodeSpec(
    ; directions=dirs, offsets=offs, featurizer=:summary,
    aggregate=:mean, normalize_weights=true, threads=true,
)
land_spec = TO.LandscapeSpec(
    ; directions=dirs, offsets=offs,
    tgrid=collect(range(0.0, stop=1.4, length=56)), kmax=3,
    aggregate=:mean, normalize_weights=true, threads=true,
)
spec = TO.CompositeSpec((sbar_spec, land_spec))

opts = TO.InvariantOptions(
    ; axes_policy=:encoding, max_axis_len=64,
      threads=true, strict=true, pl_mode=:fast,
)

println("nfeatures(spec) = ", TO.nfeatures(spec))


## Step 5: Batch featurize with SessionCache reuse


In [ ]:
samples = [(; M=encodings[i].M, pi=encodings[i].pi, label=labels[i], id=@sprintf("pc_%03d", i)) for i in eachindex(encodings)]

sc = TO.SessionCache()
fs = TO.batch_transform(
    samples,
    spec;
    opts=opts,
    idfun=s -> s.id,
    labelfun=s -> s.label,
    batch=TO.BatchOptions(threaded=true, backend=:threads, progress=false, deterministic=true),
    cache=sc,
    on_unsupported=:error,
)

println("feature matrix shape = ", size(fs.X))
println("first 5 feature names= ", TO.feature_names(spec)[1:5])


## Step 6: Export outputs


In [ ]:
wide_path = joinpath(outdir, "point_cloud_rips_density__wide.csv")
long_path = joinpath(outdir, "point_cloud_rips_density__long.csv")

TO.save_features(wide_path, fs; format=:csv, mode=:wide, metadata=true)
TO.save_features(long_path, fs; format=:csv, mode=:long, metadata=true)

pipe_path = joinpath(outdir, "pipeline.json")
TO.save_pipeline_json(
    pipe_path,
    clouds[1],
    filtration;
    degree=0,
    pipeline_opts=TO.PipelineOptions(axes_policy=:encoding, poset_kind=:signature, field=:F2),
)

println("wide csv      = ", wide_path)
println("long csv      = ", long_path)
println("pipeline json = ", pipe_path)


## Discussion prompts
- How does changing `knn` alter both runtime and statistical stability?
- Which part of this pipeline is most sensitive to `max_dim`?
- In your application, would you trust sliced summaries, landscapes, or both?
        
